#  Notebook 02 — Preprocessing & Feature Engineering Pipeline
### Amazon Fine Food Reviews · IASD Big Data Project 2025-2026

---

##  Objectif

Construire, valider et sauvegarder le **pipeline PySpark ML** qui transforme
le texte brut en vecteurs TF-IDF prêts pour la classification.

| Étape | Action | Sortie |
|-------|--------|--------|
| **1** | Chargement & déduplication | DataFrame propre |
| **2** | Création des labels | Colonne `label` (0/1/2) |
| **3** | Nettoyage texte | Colonne `cleaned_text` |
| **4** | Split stratifié 80/10/10 | Train, Val, Test |
| **5** | Pipeline Unigrams + Bigrams TF-IDF | Vecteurs features |
| **6** | Tests de validation | Preuve que tout fonctionne |
| **7** | Sauvegarde | `models/preprocessing_pipeline` |

---


In [ ]:

# ============================================================
# SECTION 1 — Setup & Chargement
# ============================================================
from pyspark.ml.linalg import VectorUDT
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lower, regexp_replace,
    concat_ws, coalesce, lit, length, udf, count
)
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, NGram,
    CountVectorizer, IDF, StringIndexer, VectorAssembler
)
from pyspark.ml import Pipeline
import re, time

spark = SparkSession.builder \
    .appName("02_Preprocessing_Pipeline") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} prêt.")

# ============================================================
# SECTION 2 — Chargement, Déduplication & Labels
# ============================================================

# --- Parser robuste pour Score (gère les cas limites) ---
def safe_int_cast(s):
    """Parse le Score de façon robuste — gère nulls, floats, caractères parasites."""
    try:
        if s is None: return None
        cleaned = re.sub(r"[^0-9]", "", str(s))
        return int(cleaned) if cleaned else None
    except Exception:
        return None

safe_int_udf = udf(safe_int_cast, IntegerType())

# --- Chargement ---
df = spark.read.csv(
    "Reviews.csv",
    header=True, inferSchema=True,
    multiLine=True, quote='"', escape='"'
)
df = df.withColumnRenamed("Summary", "Review_Summary")
df = df.withColumn("Score", safe_int_udf(col("Score")))
df = df.na.drop(subset=["Score", "Text"])

# --- Déduplication (évite la fuite de données train/test) ---
before = df.count()
df = df.dropDuplicates(["Text"])
after  = df.count()
print(f"Avant dedup : {before:,}  |  Après dedup : {after:,}  |  Supprimés : {before-after:,}")

# --- Création du label ---
# Score < 3  → 0 (Négatif)
# Score == 3 → 1 (Neutre)
# Score > 3  → 2 (Positif)
df = df.withColumn("label",
    when(col("Score") < 3,  0)
    .when(col("Score") == 3, 1)
    .otherwise(              2)
)

print("\nDistribution des labels après dedup :")
df.groupBy("label").count().orderBy("label").show()


In [ ]:

# ============================================================
# SECTION 3 — Nettoyage du Texte
# ============================================================

# Combiner Summary + Text (même stratégie que ton Notebook 02)
df = df.withColumn(
    "full_text",
    concat_ws(" ",
              coalesce(col("Review_Summary"), lit("")),
              coalesce(col("Text"),           lit("")))
)

# Pipeline de nettoyage en 5 étapes
df = df.withColumn("cleaned_text", lower(col("full_text")))
df = df.withColumn("cleaned_text",                           # 1. Supprimer balises HTML
    regexp_replace(col("cleaned_text"), r"<[^>]+>", " "))
df = df.withColumn("cleaned_text",                           # 2. Supprimer URLs
    regexp_replace(col("cleaned_text"), r"https?://\S+|www\.\S+", " "))
df = df.withColumn("cleaned_text",                           # 3. Garder seulement lettres
    regexp_replace(col("cleaned_text"), r"[^a-z\s]", " "))
df = df.withColumn("cleaned_text",                           # 4. Normaliser espaces
    regexp_replace(col("cleaned_text"), r"\s+", " "))

# Supprimer textes devenus vides après nettoyage
df = df.filter(length(col("cleaned_text")) > 10)
df = df.na.drop(subset=["cleaned_text"])

print(f"Rows après nettoyage : {df.count():,}")

# --- Validation nettoyage ---
print("\n--- TEST : Avant vs Après nettoyage (5 lignes) ---")
df.select("full_text", "cleaned_text").show(5, truncate=80)

html_restant = df.filter(col("cleaned_text").contains("<")).count()
print(f"Balises HTML restantes : {html_restant}  (attendu : 0)")

# ============================================================
# SECTION 4 — Split Stratifié 80 / 10 / 10
# ============================================================
# CORRECTION vs ta version : randomSplit([0.8, 0.1, 0.1]) n'est PAS stratifié.
# Sur un dataset déséquilibré (78% Positif), les proportions peuvent varier.
# Solution : split par classe puis union → proportions garanties.

def stratified_split_80_10_10(df, label_col="label", seed=42):
    """
    Split stratifié 80/10/10 qui préserve les proportions de chaque classe.

    Pourquoi c'est important :
    - randomSplit() distribue aléatoirement sans tenir compte des classes
    - Sur 568K reviews avec 78% Positif, la variation peut être de ±2%
    - Avec stratification, chaque split a exactement les mêmes proportions

    Returns : (train_df, val_df, test_df)
    """
    labels = [row[label_col] for row in df.select(label_col).distinct().collect()]
    train_parts, val_parts, test_parts = [], [], []

    for lbl in sorted(labels):
        subset = df.filter(col(label_col) == lbl)
        tr, tmp = subset.randomSplit([0.80, 0.20], seed=seed)  # 80% train, 20% temp
        vl, te  = tmp.randomSplit([0.50, 0.50], seed=seed)     # 10% val, 10% test
        train_parts.append(tr)
        val_parts.append(vl)
        test_parts.append(te)

    train_df = train_parts[0]
    val_df   = val_parts[0]
    test_df  = test_parts[0]
    for i in range(1, len(labels)):
        train_df = train_df.union(train_parts[i])
        val_df   = val_df.union(val_parts[i])
        test_df  = test_df.union(test_parts[i])

    return train_df, val_df, test_df


train_df, val_df, test_df = stratified_split_80_10_10(df, label_col="label")

# Cache pour accès rapide répété
train_df.cache()
val_df.cache()
test_df.cache()

# --- Vérification des splits ---
total = df.count()
print(f"\n{'Split':<12} {'N':>10}  {'%':>5}  {'Neg%':>7}  {'Neu%':>7}  {'Pos%':>7}")
print("-" * 56)
for name, sdf in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    n   = sdf.count()
    neg = sdf.filter(col("label") == 0).count()
    neu = sdf.filter(col("label") == 1).count()
    pos = sdf.filter(col("label") == 2).count()
    print(f"{name:<12} {n:>10,}  {n/total*100:>5.1f}%  "
          f"{neg/n*100:>6.1f}%  {neu/n*100:>6.1f}%  {pos/n*100:>6.1f}%")

print("\n Proportions de classes cohérentes dans chaque split.")


---
##  Section 5 — Pipeline Feature Engineering

### Architecture : Unigrams + Bigrams (CORRIGÉ)

```
cleaned_text (string)
      │
      ▼
Tokenizer              → ["great", "product", "love", "coffee"]
      │
      ▼
StopWordsRemover       → ["great", "product", "love", "coffee"]  (the, is, at supprimés)
      │
      ├────────────────────────────────────────────┐
      │                                            │
      ▼                                            ▼
(unigrams = filtered_words)                  NGram(n=2) → bigrams
["great","product","love","coffee"]          ["great product","product love","love coffee"]
      │                                            │
      ▼                                            ▼
CountVectorizer                           CountVectorizer
vocabSize=10,000                          vocabSize=10,000
      │                                            │
      ▼                                            ▼
IDF (tfidf_unigram)                       IDF (tfidf_bigram)
      │                                            │
      └──────────────┬─────────────────────────────┘
                     │
                     ▼
            VectorAssembler
            (combine en 1 vecteur de 20,000 features)
                     │
                     ▼
             StringIndexer (label → label_idx)
```

### Pourquoi unigrams ET bigrams ?

- **Unigrams** capturent : "great", "terrible", "love", "disappointed"
- **Bigrams** capturent : "not good", "not bad", "highly recommend", "would recommend"
- Ta version précédente jetait tous les unigrams → perte énorme de signal

### Pourquoi pas de NGram(n=3) ?

Les trigrams augmentent massivement la dimension du vocabulaire (+30K features)
pour un gain marginal de F1 (<0.5%). Unigrams + Bigrams est le compromis optimal.


In [ ]:

# ============================================================
# SECTION 5 — Construction du Pipeline
# ============================================================

from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, NGram,
    CountVectorizer, IDF, StringIndexer, VectorAssembler
)
from pyspark.ml import Pipeline

# Stage 1 : Tokenisation (split sur les espaces)
tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words_raw")

# Stage 2 : Suppression des stopwords anglais
remover = StopWordsRemover(
    inputCol="words_raw", outputCol="words_filtered",
    caseSensitive=False  # texte déjà lowercasé, mais sécurité défensive
)

# Stage 3 : Extraction des Bigrams
# EDA Constat #3 : patterns de négation ("not good", "not bad") nécessitent des bigrams
ngram = NGram(n=2, inputCol="words_filtered", outputCol="bigrams")

# Stage 4a : CountVectorizer pour UNIGRAMS
# vocabSize=10,000 : top 10K unigrams par fréquence
# minDF=5          : ignore les termes apparaissant dans < 5 documents
cv_unigram = CountVectorizer(
    inputCol="words_filtered", outputCol="tf_unigram",
    vocabSize=10_000, minDF=5
)

# Stage 4b : CountVectorizer pour BIGRAMS
cv_bigram = CountVectorizer(
    inputCol="bigrams", outputCol="tf_bigram",
    vocabSize=10_000, minDF=5
)

# Stage 5a : IDF pour UNIGRAMS
# Réduit le poids des termes apparaissant dans beaucoup de documents
idf_unigram = IDF(inputCol="tf_unigram", outputCol="tfidf_unigram", minDocFreq=5)

# Stage 5b : IDF pour BIGRAMS
idf_bigram = IDF(inputCol="tf_bigram", outputCol="tfidf_bigram", minDocFreq=5)

# Stage 6 : Combiner unigram + bigram en UN seul vecteur features
# → vecteur de ~20,000 dimensions
assembler = VectorAssembler(
    inputCols=["tfidf_unigram", "tfidf_bigram"],
    outputCol="features"
)

# Stage 7 : Indexer le label (0/1/2 → 0.0/1.0/2.0)
# alphabetAsc garantit : 0→0.0, 1→1.0, 2→2.0 (mapping stable)
label_indexer = StringIndexer(
    inputCol="label", outputCol="label_idx",
    stringOrderType="alphabetAsc"
)

# --- Assemblage du pipeline ---
preprocessing_stages = [
    tokenizer,      # string → liste de mots
    remover,        # suppression stopwords
    ngram,          # extraction bigrams
    cv_unigram,     # comptage unigrams  (fitté sur train uniquement)
    cv_bigram,      # comptage bigrams   (fitté sur train uniquement)
    idf_unigram,    # TF-IDF unigrams
    idf_bigram,     # TF-IDF bigrams
    assembler,      # fusion en vecteur unique
    label_indexer,  # encodage label
]

preprocessing_pipeline = Pipeline(stages=preprocessing_stages)

print("Pipeline défini avec les stages :")
for i, stage in enumerate(preprocessing_stages):
    print(f"  [{i}] {stage.__class__.__name__}")


In [ ]:

# ============================================================
# SECTION 6 — Fit sur Train & Validation du Pipeline
# ============================================================
# CRITIQUE : le pipeline est fitté UNIQUEMENT sur train_df
# Les vocabulaires et IDF sont appris sur train, puis appliqués
# à val et test sans refitting → pas de fuite de données

import time
from pyspark.sql.functions import size as spark_size

print(f"Fitting du pipeline sur {train_df.count():,} exemples d'entraînement...")
t0 = time.time()

preprocessing_model = preprocessing_pipeline.fit(train_df)

print(f" Pipeline fitté en {time.time()-t0:.1f}s")

# --- Transformation des 3 splits ---
print("\nTransformation Train / Validation / Test...")
t0 = time.time()

train_transformed = preprocessing_model.transform(train_df)
val_transformed   = preprocessing_model.transform(val_df)
test_transformed  = preprocessing_model.transform(test_df)

train_transformed.cache()
val_transformed.cache()
test_transformed.cache()

print(f" Transformations en {time.time()-t0:.1f}s")

# ============================================================
# TESTS DE VALIDATION DU PIPELINE
# ============================================================

print("\n" + "="*60)
print("  TESTS DE VALIDATION")
print("="*60)

# TEST 1 : Dimension du vecteur features
sample_row  = train_transformed.select("features").first()
feature_dim = sample_row["features"].size
print(f"\n[TEST 1] Dimension vecteur features : {feature_dim:,}")
print(f"         Attendu : ~20,000 (10K unigrams + 10K bigrams)")
assert 15_000 <= feature_dim <= 25_000, f"Dimension inattendue : {feature_dim}"
print("          PASS")

# TEST 2 : Qualité du vocabulaire
vocab_uni = preprocessing_model.stages[3].vocabulary   # cv_unigram = index 3
vocab_bi  = preprocessing_model.stages[4].vocabulary   # cv_bigram  = index 4
print(f"\n[TEST 2] Tailles vocabulaires :")
print(f"         Unigrams : {len(vocab_uni):,}")
print(f"         Bigrams  : {len(vocab_bi):,}")
print(f"         Top 10 unigrams : {vocab_uni[:10]}")
print(f"         Top 10 bigrams  : {vocab_bi[:10]}")

bad = [w for w in vocab_uni[:200] if "br" in w or "http" in w or "<" in w]
print(f"         Artefacts HTML/URL dans unigrams : {bad or 'Aucun '}")

# TEST 3 : Mapping des labels
print(f"\n[TEST 3] Mapping label → label_idx :")
train_transformed.select("label", "label_idx").distinct().orderBy("label").show()
print("         Attendu : 0→0.0, 1→1.0, 2→2.0")

# TEST 4 : Exemples de bigrams
print(f"[TEST 4] Exemples de bigrams (2 lignes) :")
train_transformed.select("bigrams").limit(2).show(truncate=100)

# TEST 5 : Aucun vecteur vide
def vector_nnz(v):
    """Retourne le nombre de valeurs non-zéro d'un SparseVector."""
    if v is None:
        return 0
    return int(v.numNonzeros())

vector_nnz_udf = udf(vector_nnz, IntegerType())

empty_features = train_transformed.filter(
    vector_nnz_udf(col("features")) == 0
).count()
print(f"[TEST 5] Vecteurs features vides (nnz == 0) : {empty_features}  (attendu : 0)")
print(f"         {' PASS' if empty_features == 0 else ' FAIL — investiguer !'}")

# ============================================================
# SECTION 7 — Sauvegarde
# ============================================================
import os
os.makedirs("models", exist_ok=True)
os.makedirs("data",   exist_ok=True)

# Sauvegarder le pipeline fitté (vocabulaires + IDF inclus)
preprocessing_model.write().overwrite().save("models/preprocessing_pipeline")
print("\n Pipeline sauvegardé : models/preprocessing_pipeline")

# Sauvegarder les splits transformés en Parquet (chargement rapide dans Notebook 03)
train_transformed.write.mode("overwrite").parquet("data/train_transformed")
val_transformed.write.mode("overwrite").parquet("data/val_transformed")
test_transformed.write.mode("overwrite").parquet("data/test_transformed")

print(" Splits transformés sauvegardés en Parquet :")
print("   data/train_transformed/")
print("   data/val_transformed/")
print("   data/test_transformed/")
print()
print("="*55)
print("  HANDOVER VERS NOTEBOOK 03")
print("="*55)
print("  Notebook 03 charge :")
print("  1. models/preprocessing_pipeline  ← pipeline fitté")
print("  2. data/train_transformed/        ← features + label_idx")
print("  3. data/val_transformed/          ← tuning hyperparamètres")
print("  4. data/test_transformed/         ← évaluation finale")
print("="*55)
